# Columns Check

In [2]:
# Script to List Available Fields in NHTSA Crash Viewer API (Fixed for CrashResultSet)

import requests
import json
import logging

# ========== CONFIGURATION ==========
NHTSA_API_BASE = "https://crashviewer.nhtsa.dot.gov/CrashAPI"
STATE = 1  # Alabama
FROM_YEAR = 2022
TO_YEAR = 2022
FORMAT = "json"
HEADERS = {
    "X-Forwarded-For": "8.8.8.8",  # Simulated US IP
    "User-Agent": "Mozilla/5.0"
}
MIN_VEHICLES = 1
MAX_VEHICLES = 99

# ========== SET UP LOGGING ==========
logging.basicConfig(
    filename="api_fields.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ========== FUNCTION TO FETCH CRASH LIST ==========
def fetch_crash_case_list(state, from_year, to_year):
    url = f"{NHTSA_API_BASE}/crashes/GetCaseList"
    params = {
        "states": state,
        "fromYear": from_year,
        "toYear": to_year,
        "minNumOfVehicles": MIN_VEHICLES,
        "maxNumOfVehicles": MAX_VEHICLES,
        "format": FORMAT
    }

    try:
        response = requests.get(url, params=params, headers=HEADERS)
        response.raise_for_status()
        data = response.json()

        # Handle 'Results' list
        if isinstance(data, dict) and "Results" in data:
            crashes = data["Results"]
            if isinstance(crashes, list) and crashes and isinstance(crashes[0], list):
                return crashes[0]  # Flatten nested list
            elif isinstance(crashes, list):
                return crashes

        # Handle nested list
        if isinstance(data, list) and len(data) == 1 and isinstance(data[0], list):
            return data[0]

        raise ValueError("Unexpected response format")
    except Exception as e:
        logging.error(f"Error fetching crash list: {e}")
        print(f"❌ Error fetching crash list: {e}")
        return []

# ========== FUNCTION TO FETCH CRASH DETAILS ==========
def fetch_crash_details(case_year, state, state_case_id):
    url = f"{NHTSA_API_BASE}/crashes/GetCaseDetails"
    params = {
        "caseYear": case_year,
        "state": state,
        "stateCase": state_case_id,
        "format": FORMAT
    }

    try:
        response = requests.get(url, params=params, headers=HEADERS)
        response.raise_for_status()
        data = response.json()
        return response.json()
        return data
    except Exception as e:
        logging.error(f"Error fetching details for ID {state_case_id}: {e}")
        print(f"❌ Error fetching details for ID {state_case_id}: {e}")
        return {}

# ========== MAIN FUNCTION TO LIST FIELDS ==========
def list_available_fields():
    print("📥 Fetching crash list...")
    logging.info("Starting field discovery...")
    
    try:
        crashes = fetch_crash_case_list(STATE, FROM_YEAR, TO_YEAR)
        if not crashes:
            print("⚠️ No crashes found.")
            logging.warning("No crashes found.")
            return

        # Process only the first crash
        crash = crashes[0]
        state_case_id = crash.get("St_Case") or crash.get("st_case")
        case_year = crash.get("case_year") or FROM_YEAR

        if not state_case_id:
            print("⚠️ Missing crash ID.")
            logging.warning(f"Missing crash ID in: {crash}")
            return

        # Get fields from GetCaseList
        case_list_fields = list(crash.keys())
        print(f"✔️ GetCaseList fields: {case_list_fields}")
        logging.info(f"GetCaseList fields: {case_list_fields}")

        # Fetch and get fields from GetCaseDetails
        print(f"🔍 Fetching details for Crash ID: {state_case_id}")
        details = fetch_crash_details(case_year, STATE, state_case_id)

        # Save raw response to file
        with open(f"crash_details_{state_case_id}.json", "w") as f:
            json.dump(details, f, indent=2)

        # Log raw response
        logging.info(f"Raw GetCaseDetails response for {state_case_id}: {json.dumps(details, indent=2)}")

        details_data = details
        if "Results" in details and isinstance(details["Results"], list) and details["Results"]:
            results = details["Results"][0]
            if isinstance(results, list) and results and isinstance(results[0], dict):
                details_data = results[0]  # Handle list of dictionaries
            elif isinstance(results, dict):
                details_data = results  # Direct dictionary
            else:
                print(f"⚠️ Unexpected Results[0] type: {type(results)}")
                logging.warning(f"Unexpected Results[0] type: {type(results)}")
                return

        # Check for CrashResultSet
        if "CrashResultSet" in details_data:
            crash_result_set = details_data["CrashResultSet"]
            if isinstance(crash_result_set, list) and crash_result_set and isinstance(crash_result_set[0], dict):
                details_data = crash_result_set[0]  # Take first dictionary
            elif isinstance(crash_result_set, dict):
                details_data = crash_result_set
            else:
                print(f"⚠️ Unexpected CrashResultSet type: {type(crash_result_set)}")
                logging.warning(f"Unexpected CrashResultSet type: {type(crash_result_set)}")
                return

        details_fields = list(details_data.keys()) if isinstance(details_data, dict) else []
        print(f"✔️ GetCaseDetails fields: {details_fields}")
        logging.info(f"GetCaseDetails fields: {details_fields}")

        # Save fields to file
        with open("available_fields.txt", "w") as f:
            f.write("GetCaseList Fields:\n")
            f.write(", ".join(case_list_fields) + "\n\n")
            f.write("GetCaseDetails Fields:\n")
            f.write(", ".join(details_fields) + "\n")

        print("✅ Available fields saved to 'available_fields.txt'")
        logging.info("Available fields saved to 'available_fields.txt'")
    except Exception as e:
        print(f"❌ Error: {e}")
        logging.error(f"Error: {e}")

# ========== ENTRY POINT ==========
if __name__ == "__main__":
    list_available_fields()

📥 Fetching crash list...
✔️ GetCaseList fields: ['CountyName', 'CrashDate', 'Fatals', 'Peds', 'Persons', 'St_Case', 'State', 'StateName', 'TotalVehicles']
🔍 Fetching details for Crash ID: 10012
✔️ GetCaseDetails fields: ['ARR_HOUR', 'ARR_HOURNAME', 'ARR_MIN', 'ARR_MINNAME', 'CEvents', 'CF1', 'CF1NAME', 'CF2', 'CF2NAME', 'CF3', 'CF3NAME', 'CITY', 'CITYNAME', 'COUNTY', 'COUNTYNAME', 'CaseYear', 'CrashRFs', 'DAY', 'DAYNAME', 'DAY_WEEK', 'DAY_WEEKNAME', 'DRUNK_DR', 'FATALS', 'FUNC_SYS', 'FUNC_SYSNAME', 'HARM_EV', 'HARM_EVNAME', 'HOSP_HR', 'HOSP_HRNAME', 'HOSP_MN', 'HOSP_MNNAME', 'HOUR', 'HOURNAME', 'LATITUDE', 'LATITUDENAME', 'LGT_COND', 'LGT_CONDNAME', 'LONGITUD', 'LONGITUDNAME', 'MAN_COLL', 'MAN_COLLNAME', 'MILEPT', 'MILEPTNAME', 'MINUTE', 'MINUTENAME', 'MONTH', 'MonthName', 'NHS', 'NHSNAME', 'NMDrugs', 'NMPersonRF', 'NMRace', 'NOT_HOUR', 'NOT_HOURNAME', 'NOT_MIN', 'NOT_MINNAME', 'NPersons', 'NmCrashes', 'NmDistract', 'NmImpairs', 'NmPriors', 'PEDS', 'PERMVIT', 'PERNOTMVIT', 'PERSONS', '

# Testing 1

In [ ]:
# US Crash Data Collection Script (Fixed Version for NHTSA API)

import requests
import csv
import time
import json

# ========== CONFIGURATION ==========
NHTSA_API_BASE = "https://crashviewer.nhtsa.dot.gov/CrashAPI"
STATE = 1  # Alabama
FROM_YEAR = 2022
TO_YEAR = 2022
FORMAT = "json"
HEADERS = {
    "X-Forwarded-For": "8.8.8.8",  # Simulated US IP
    "User-Agent": "Mozilla/5.0"
}
MIN_VEHICLES = 1
MAX_VEHICLES = 99

# ========== FUNCTION TO FETCH CRASH LIST ==========
def fetch_crash_case_list(state, from_year, to_year):
    url = f"{NHTSA_API_BASE}/crashes/GetCaseList"
    params = {
        "states": state,
        "fromYear": from_year,
        "toYear": to_year,
        "minNumOfVehicles": MIN_VEHICLES,
        "maxNumOfVehicles": MAX_VEHICLES,
        "format": FORMAT
    }

    response = requests.get(url, params=params, headers=HEADERS)
    response.raise_for_status()
    data = response.json()

    # CASE 1: Response contains a 'Results' list
    if isinstance(data, dict) and "Results" in data:
        crashes = data["Results"]
        if isinstance(crashes, list) and crashes and isinstance(crashes[0], list):
            # Flatten nested list
            flat_crashes = crashes[0]
            print(f"✔️ Found {len(flat_crashes)} crash records (via 'Results' nested list).")
            return flat_crashes
        elif isinstance(crashes, list) and all(isinstance(item, dict) for item in crashes):
            print(f"✔️ Found {len(crashes)} crash records (via 'Results').")
            return crashes

    # CASE 2: Response is a nested list [[dict, dict, ...]]
    if isinstance(data, list) and len(data) == 1 and isinstance(data[0], list):
        flat_data = data[0]
        print(f"✔️ Found {len(flat_data)} crash records (nested list).")
        return flat_data

    # CASE 3: Unexpected structure
    raise ValueError("❌ Unexpected response format from API: " + json.dumps(data, indent=2))

# ========== FUNCTION TO FETCH CRASH DETAILS ==========
def fetch_crash_details(case_year, state, state_case_id):
    url = f"{NHTSA_API_BASE}/crashes/GetCaseDetails"
    params = {
        "caseYear": case_year,
        "state": state,
        "stateCase": state_case_id,
        "format": FORMAT
    }

    response = requests.get(url, params=params, headers=HEADERS)
    response.raise_for_status()
    data = response.json()

    if not data or not isinstance(data, dict):
        raise ValueError(f"❌ No valid crash details for ID {state_case_id}")
    return data

# ========== MAIN PIPELINE FUNCTION ==========
def run_pipeline():
    print("📥 Fetching crash list...")
    try:
        crashes = fetch_crash_case_list(STATE, FROM_YEAR, TO_YEAR)
    except Exception as e:
        print(f"❌ Error fetching crash list: {e}")
        return

    output = []

    for crash in crashes:
        if not isinstance(crash, dict):
            print(f"⚠️ Skipping non-dict crash item: {crash}")
            continue

        state_case_id = crash.get("St_Case") or crash.get("st_case")
        case_year = crash.get("case_year") or FROM_YEAR  # Fallback to FROM_YEAR

        if not state_case_id:
            print(f"⚠️ Missing crash ID in: {crash}")
            continue

        try:
            print(f"🔍 Fetching details for Crash ID: {state_case_id} (Year: {case_year})")
            details = fetch_crash_details(case_year, STATE, state_case_id)

            # DEBUG: Pretty-print the entire response
            print(f"📦 API Response for {state_case_id}:\n{json.dumps(details, indent=2)}")

            # Navigate the response structure
            crash_data = details
            if "Results" in details and isinstance(details["Results"], list) and details["Results"]:
                crash_data = details["Results"][0]

            # Try possible field names based on FARS data
            lat = (crash_data.get("LATITUDE") or crash_data.get("latitude") or
                   crash_data.get("Latitude") or crash_data.get("LAT"))
            lon = (crash_data.get("LONGITUDE") or crash_data.get("longitude") or
                   crash_data.get("Longitude") or crash_data.get("LON"))
            date = (crash_data.get("CRASH_DATE") or crash_data.get("crash_date") or
                    crash_data.get("CrashDate") or crash_data.get("CRASHDATE"))

            # Validate required fields
            if not all([lat, lon, date]):
                print(f"⚠️ Incomplete data for crash ID {state_case_id}: lat={lat}, lon={lon}, date={date}")
                continue

            record = {
                "Crash ID": state_case_id,
                "Date": date,
                "Latitude": lat,
                "Longitude": lon
            }
            output.append(record)
            print(f"✅ Processed crash ID {state_case_id}")
            time.sleep(1)  # Respect API rate limits
        except Exception as e:
            print(f"❌ Failed to fetch details for ID {state_case_id}: {e}")

    if output:
        with open("us_crash_data.csv", "w", newline='') as file:
            writer = csv.DictWriter(file, fieldnames=output[0].keys())
            writer.writeheader()
            writer.writerows(output)
        print(f"✅ Data successfully written to 'us_crash_data.csv' with {len(output)} records")
    else:
        print("⚠️ No crash records were processed successfully.")

# ========== ENTRY POINT ==========
if __name__ == "__main__":
    run_pipeline()

# TEST CODE FOR ONE STATE ALL THE COLUMNS


In [1]:
# US Crash Data Collection Script (Fixed Version for NHTSA API)

import requests
import csv
import time
import json
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.exceptions import HTTPError
import logging

# ========== CONFIGURATION ==========
NHTSA_API_BASE = "https://crashviewer.nhtsa.dot.gov/CrashAPI"
STATE = 1  # Alabama
FROM_YEAR = 2022
TO_YEAR = 2022
FORMAT = "json"
HEADERS = {
    "X-Forwarded-For": "8.8.8.8",  # Simulated US IP
    "User-Agent": "Mozilla/5.0"
}
MIN_VEHICLES = 1
MAX_VEHICLES = 99
MAX_WORKERS = 10  # Number of concurrent requests

# Set up logging
logging.basicConfig(filename="crash_data.log", level=logging.INFO)

# ========== FUNCTION TO FETCH CRASH LIST ==========
def fetch_crash_case_list(state, from_year, to_year):
    url = f"{NHTSA_API_BASE}/crashes/GetCaseList"
    params = {
        "states": state,
        "fromYear": from_year,
        "toYear": to_year,
        "minNumOfVehicles": MIN_VEHICLES,
        "maxNumOfVehicles": MAX_VEHICLES,
        "format": FORMAT
    }

    response = requests.get(url, params=params, headers=HEADERS)
    response.raise_for_status()
    data = response.json()

    # CASE 1: Response contains a 'Results' list
    if isinstance(data, dict) and "Results" in data:
        crashes = data["Results"]
        if isinstance(crashes, list) and crashes and isinstance(crashes[0], list):
            # Flatten nested list
            flat_crashes = crashes[0]
            return flat_crashes
        elif isinstance(crashes, list) and all(isinstance(item, dict) for item in crashes):
            return crashes

    # CASE 2: Response is a nested list [[dict, dict, ...]]
    if isinstance(data, list) and len(data) == 1 and isinstance(data[0], list):
        flat_data = data[0]
        return flat_data

    # CASE 3: Unexpected structure
    raise ValueError("Unexpected response format from API")

# ========== FUNCTION TO FETCH CRASH DETAILS ==========
def fetch_crash_details(case_year, state, state_case_id, retries=3, delay=0.5):
    url = f"{NHTSA_API_BASE}/crashes/GetCaseDetails"
    params = {
        "caseYear": case_year,
        "state": state,
        "stateCase": state_case_id,
        "format": FORMAT
    }

    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, headers=HEADERS)
            response.raise_for_status()
            data = response.json()

            if not data or not isinstance(data, dict):
                raise ValueError(f"No valid crash details for ID {state_case_id}")
            return data
        except HTTPError as e:
            if e.response.status_code == 429:  # Rate limit exceeded
                if attempt < retries - 1:
                    time.sleep(delay)
                    continue
            raise
        except Exception as e:
            logging.error(f"Error fetching details for {state_case_id}: {e}")
            raise
    raise ValueError(f"Failed to fetch details for ID {state_case_id} after {retries} attempts")

# ========== FUNCTION TO PROCESS A SINGLE CRASH ==========
def process_crash(crash):
    if not isinstance(crash, dict):
        return None

    state_case_id = crash.get("St_Case") or crash.get("st_case")
    case_year = crash.get("case_year") or FROM_YEAR  # Fallback to FROM_YEAR

    if not state_case_id:
        return None

    try:
        details = fetch_crash_details(case_year, STATE, state_case_id)

        # Navigate the response structure
        crash_data = details
        if "Results" in details and isinstance(details["Results"], list) and details["Results"]:
            results = details["Results"][0]
            if isinstance(results, list) and results and isinstance(results[0], dict):
                crash_data = results[0]  # Handle list of dictionaries
            elif isinstance(results, dict):
                crash_data = results  # Direct dictionary
            else:
                crash_data = {}

        # Check for CrashResultSet
        if "CrashResultSet" in crash_data:
            crash_result_set = crash_data["CrashResultSet"]
            if isinstance(crash_result_set, list) and crash_result_set and isinstance(crash_result_set[0], dict):
                crash_data = crash_result_set[0]  # Take first dictionary
            elif isinstance(crash_result_set, dict):
                crash_data = crash_result_set
            else:
                crash_data = {}

        # Log extracted crash_data for debugging
        logging.info(f"Extracted crash_data for {state_case_id}: {json.dumps(crash_data, indent=2)}")

        # Construct CRASH_DATE from YEAR, MONTH, DAY, HOUR, MINUTE with variants
        crash_date = (str(crash_data.get("YEAR", crash_data.get("year", ""))) + "-" + 
                      str(crash_data.get("MONTH", crash_data.get("month", ""))).zfill(2) + "-" + 
                      str(crash_data.get("DAY", crash_data.get("day", ""))).zfill(2) + " " + 
                      str(crash_data.get("HOUR", crash_data.get("hour", ""))).zfill(2) + ":" + 
                      str(crash_data.get("MINUTE", crash_data.get("minute", ""))).zfill(2))

        # Create record with specified fields, trying multiple field name variants
        record = {
            # GetCaseList fields
            "Crash ID": state_case_id,
            "CrashDate": crash.get("CrashDate", ""),
            "CountyName": crash.get("CountyName", ""),
            "Fatals": crash.get("Fatals", ""),
            "Peds": crash.get("Peds", ""),
            "Persons": crash.get("Persons", ""),
            "State": crash.get("State", ""),
            "StateName": crash.get("StateName", ""),
            "TotalVehicles": crash.get("TotalVehicles", ""),
            # GetCaseDetails fields with variants
            "Crash Date": crash_date,
            "Latitude": crash_data.get("LATITUDE", crash_data.get("latitude", "")),
            "Longitude": crash_data.get("LONGITUD", crash_data.get("longitude", crash_data.get("LONGITUDE", ""))),
            "Weather": crash_data.get("WEATHER", crash_data.get("weather", "")),
            "Weather Name": crash_data.get("WEATHERNAME", crash_data.get("weatherName", "")),
            "Road Function": crash_data.get("ROAD_FNC", crash_data.get("road_fnc", "")),
            "Road Function Name": crash_data.get("ROAD_FNCNAME", crash_data.get("road_fncName", "")),
            "Light Condition": crash_data.get("LGT_COND", crash_data.get("lgt_cond", "")),
            "Light Condition Name": crash_data.get("LGT_CONDNAME", crash_data.get("lgt_condName", "")),
            "Manner of Collision": crash_data.get("MAN_COLL", crash_data.get("man_coll", "")),
            "Manner of Collision Name": crash_data.get("MAN_COLLNAME", crash_data.get("man_collName", "")),
            "Speed Limit": crash_data.get("SP_JUR", crash_data.get("sp_jur", "")),
            "Speed Limit Name": crash_data.get("SP_JURNAME", crash_data.get("sp_jurName", "")),
            "Harmful Event": crash_data.get("HARM_EV", crash_data.get("harm_ev", "")),
            "Harmful Event Name": crash_data.get("HARM_EVNAME", crash_data.get("harm_evName", "")),
            "Drunk Drivers": crash_data.get("DRUNK_DR", crash_data.get("drunk_dr", "")),
            "City": crash_data.get("CITY", crash_data.get("city", "")),
            "City Name": crash_data.get("CITYNAME", crash_data.get("cityName", "")),
            "Functional System": crash_data.get("FUNC_SYS", crash_data.get("func_sys", "")),
            "Functional System Name": crash_data.get("FUNC_SYSNAME", crash_data.get("func_sysName", "")),
            "Work Zone": crash_data.get("WRK_ZONE", crash_data.get("wrk_zone", "")),
            "Work Zone Name": crash_data.get("WRK_ZONENAME", crash_data.get("wrk_zoneName", ""))
        }
        return record
    except Exception as e:
        logging.error(f"Error processing crash {state_case_id}: {e}")
        return None

# ========== MAIN PIPELINE FUNCTION ==========
def run_pipeline():
    try:
        crashes = fetch_crash_case_list(STATE, FROM_YEAR, TO_YEAR)
    except Exception as e:
        logging.error(f"Error fetching crash list: {e}")
        return

    output = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_crash = {executor.submit(process_crash, crash): crash for crash in crashes}
        for future in tqdm(as_completed(future_to_crash), total=len(crashes), desc="Processing crashes"):
            record = future.result()
            if record:
                output.append(record)

    if output:
        with open("us_crash_data.csv", "w", newline='') as file:
            writer = csv.DictWriter(file, fieldnames=output[0].keys())
            writer.writeheader()
            writer.writerows(output)

# ========== ENTRY POINT ==========
if __name__ == "__main__":
    run_pipeline()

Processing crashes: 100%|██████████| 913/913 [04:14<00:00,  3.59it/s]


# FINAL TESTING PROPERLY MAPPED

In [2]:
# US Crash Data Collection Script (Fixed Version for NHTSA API)

import requests
import csv
import time
import json
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.exceptions import HTTPError
import logging

# ========== CONFIGURATION ==========
NHTSA_API_BASE = "https://crashviewer.nhtsa.dot.gov/CrashAPI"
STATE = 1  # Alabama
FROM_YEAR = 2022
TO_YEAR = 2022
FORMAT = "json"
HEADERS = {
    "X-Forwarded-For": "8.8.8.8",  # Simulated US IP
    "User-Agent": "Mozilla/5.0"
}
MIN_VEHICLES = 1
MAX_VEHICLES = 99
MAX_WORKERS = 10  # Number of concurrent requests

# Set up logging
logging.basicConfig(filename="crash_data.log", level=logging.INFO)

# ========== FUNCTION TO FETCH CRASH LIST ==========
def fetch_crash_case_list(state, from_year, to_year):
    url = f"{NHTSA_API_BASE}/crashes/GetCaseList"
    params = {
        "states": state,
        "fromYear": from_year,
        "toYear": to_year,
        "minNumOfVehicles": MIN_VEHICLES,
        "maxNumOfVehicles": MAX_VEHICLES,
        "format": FORMAT
    }

    response = requests.get(url, params=params, headers=HEADERS)
    response.raise_for_status()
    data = response.json()

    # CASE 1: Response contains a 'Results' list
    if isinstance(data, dict) and "Results" in data:
        crashes = data["Results"]
        if isinstance(crashes, list) and crashes and isinstance(crashes[0], list):
            # Flatten nested list
            flat_crashes = crashes[0]
            return flat_crashes
        elif isinstance(crashes, list) and all(isinstance(item, dict) for item in crashes):
            return crashes

    # CASE 2: Response is a nested list [[dict, dict, ...]]
    if isinstance(data, list) and len(data) == 1 and isinstance(data[0], list):
        flat_data = data[0]
        return flat_data

    # CASE 3: Unexpected structure
    raise ValueError("Unexpected response format from API")

# ========== FUNCTION TO FETCH CRASH DETAILS ==========
def fetch_crash_details(case_year, state, state_case_id, retries=3, delay=0.5):
    url = f"{NHTSA_API_BASE}/crashes/GetCaseDetails"
    params = {
        "caseYear": case_year,
        "state": state,
        "stateCase": state_case_id,
        "format": FORMAT
    }

    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, headers=HEADERS)
            response.raise_for_status()
            data = response.json()

            if not data or not isinstance(data, dict):
                raise ValueError(f"No valid crash details for ID {state_case_id}")
            return data
        except HTTPError as e:
            if e.response.status_code == 429:  # Rate limit exceeded
                if attempt < retries - 1:
                    time.sleep(delay)
                    continue
            raise
        except Exception as e:
            logging.error(f"Error fetching details for {state_case_id}: {e}")
            raise
    raise ValueError(f"Failed to fetch details for ID {state_case_id} after {retries} attempts")

# ========== FUNCTION TO PROCESS A SINGLE CRASH ==========
def process_crash(crash):
    if not isinstance(crash, dict):
        return None

    state_case_id = crash.get("St_Case") or crash.get("st_case")
    case_year = crash.get("case_year") or FROM_YEAR  # Fallback to FROM_YEAR

    if not state_case_id:
        return None

    try:
        details = fetch_crash_details(case_year, STATE, state_case_id)

        # Navigate to CrashResultSet
        crash_data = {}
        if "Results" in details and isinstance(details["Results"], list) and details["Results"]:
            results = details["Results"][0]
            if isinstance(results, list) and results and isinstance(results[0], dict) and "CrashResultSet" in results[0]:
                crash_data = results[0]["CrashResultSet"]

        # Construct CRASH_DATE from YEAR, MONTH, DAY, HOUR, MINUTE
        crash_date = (crash_data.get("YEAR", "") + "-" + 
                      str(crash_data.get("MONTH", "")).zfill(2) + "-" + 
                      str(crash_data.get("DAY", "")).zfill(2) + " " + 
                      str(crash_data.get("HOUR", "")).zfill(2) + ":" + 
                      str(crash_data.get("MINUTE", "")).zfill(2))

        # Create record with specified fields mapped to JSON response
        record = {
            # GetCaseList fields
            "Crash ID": state_case_id,
            "CrashDate": crash.get("CrashDate", ""),
            "CountyName": crash.get("CountyName", ""),
            "Fatals": crash.get("Fatals", ""),
            "Peds": crash.get("Peds", ""),
            "Persons": crash.get("Persons", ""),
            "State": crash.get("State", ""),
            "StateName": crash.get("StateName", ""),
            "TotalVehicles": crash.get("TotalVehicles", ""),
            # GetCaseDetails fields
            "Crash Date": crash_date,
            "Latitude": crash_data.get("LATITUDE", ""),
            "Longitude": crash_data.get("LONGITUD", ""),
            "Weather": crash_data.get("WEATHER", ""),
            "Weather Name": crash_data.get("WEATHERNAME", ""),
            "Road Function": crash_data.get("ROAD_FNC", ""),
            "Road Function Name": crash_data.get("ROAD_FNCNAME", ""),
            "Light Condition": crash_data.get("LGT_COND", ""),
            "Light Condition Name": crash_data.get("LGT_CONDNAME", ""),
            "Manner of Collision": crash_data.get("MAN_COLL", ""),
            "Manner of Collision Name": crash_data.get("MAN_COLLNAME", ""),
            "Speed Limit": crash_data.get("SP_JUR", ""),
            "Speed Limit Name": crash_data.get("SP_JURNAME", ""),
            "Harmful Event": crash_data.get("HARM_EV", ""),
            "Harmful Event Name": crash_data.get("HARM_EVNAME", ""),
            "Drunk Drivers": crash_data.get("DRUNK_DR", ""),
            "City": crash_data.get("CITY", ""),
            "City Name": crash_data.get("CITYNAME", ""),
            "Functional System": crash_data.get("FUNC_SYS", ""),
            "Functional System Name": crash_data.get("FUNC_SYSNAME", ""),
            "Work Zone": crash_data.get("WRK_ZONE", ""),
            "Work Zone Name": crash_data.get("WRK_ZONENAME", "")
        }
        return record
    except Exception as e:
        logging.error(f"Error processing crash {state_case_id}: {e}")
        return None

# ========== MAIN PIPELINE FUNCTION ==========
def run_pipeline():
    try:
        crashes = fetch_crash_case_list(STATE, FROM_YEAR, TO_YEAR)
    except Exception as e:
        logging.error(f"Error fetching crash list: {e}")
        return

    output = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_crash = {executor.submit(process_crash, crash): crash for crash in crashes}
        for future in tqdm(as_completed(future_to_crash), total=len(crashes), desc="Processing crashes"):
            record = future.result()
            if record:
                output.append(record)

    if output:
        with open("us_crash_data.csv", "w", newline='') as file:
            writer = csv.DictWriter(file, fieldnames=output[0].keys())
            writer.writeheader()
            writer.writerows(output)

# ========== ENTRY POINT ==========
if __name__ == "__main__":
    run_pipeline()

Processing crashes: 100%|██████████| 913/913 [03:45<00:00,  4.05it/s]


# All states Data from 2015 to 2025 USA 

In [3]:
# US Crash Data Collection Script (Fixed Version for NHTSA API)

import requests
import csv
import time
import json
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.exceptions import HTTPError
import logging

# ========== CONFIGURATION ==========
NHTSA_API_BASE = "https://crashviewer.nhtsa.dot.gov/CrashAPI"
FROM_YEAR = 2015
TO_YEAR = 2025
FORMAT = "json"
HEADERS = {
    "X-Forwarded-For": "8.8.8.8",  # Simulated US IP
    "User-Agent": "Mozilla/5.0"
}
MIN_VEHICLES = 1
MAX_VEHICLES = 99
MAX_WORKERS = 10  # Number of concurrent requests

# List of all U.S. states and territories with NHTSA codes
STATES = [
    (1, "Alabama"), (2, "Alaska"), (4, "Arizona"), (5, "Arkansas"), (6, "California"),
    (8, "Colorado"), (9, "Connecticut"), (10, "Delaware"), (11, "District_of_Columbia"),
    (12, "Florida"), (13, "Georgia"), (15, "Hawaii"), (16, "Idaho"), (17, "Illinois"),
    (18, "Indiana"), (19, "Iowa"), (20, "Kansas"), (21, "Kentucky"), (22, "Louisiana"),
    (23, "Maine"), (24, "Maryland"), (25, "Massachusetts"), (26, "Michigan"), (27, "Minnesota"),
    (28, "Mississippi"), (29, "Missouri"), (30, "Montana"), (31, "Nebraska"), (32, "Nevada"),
    (33, "New_Hampshire"), (34, "New_Jersey"), (35, "New_Mexico"), (36, "New_York"),
    (37, "North_Carolina"), (38, "North_Dakota"), (39, "Ohio"), (40, "Oklahoma"), (41, "Oregon"),
    (42, "Pennsylvania"), (44, "Rhode_Island"), (45, "South_Carolina"), (46, "South_Dakota"),
    (47, "Tennessee"), (48, "Texas"), (49, "Utah"), (50, "Vermont"), (51, "Virginia"),
    (53, "Washington"), (54, "West_Virginia"), (55, "Wisconsin"), (56, "Wyoming"),
    (72, "Puerto_Rico")
]

# Set up logging
logging.basicConfig(filename="crash_data.log", level=logging.INFO)

# ========== FUNCTION TO FETCH CRASH LIST ==========
def fetch_crash_case_list(state, from_year, to_year):
    url = f"{NHTSA_API_BASE}/crashes/GetCaseList"
    params = {
        "states": state,
        "fromYear": from_year,
        "toYear": to_year,
        "minNumOfVehicles": MIN_VEHICLES,
        "maxNumOfVehicles": MAX_VEHICLES,
        "format": FORMAT
    }

    response = requests.get(url, params=params, headers=HEADERS)
    response.raise_for_status()
    data = response.json()

    # CASE 1: Response contains a 'Results' list
    if isinstance(data, dict) and "Results" in data:
        crashes = data["Results"]
        if isinstance(crashes, list) and crashes and isinstance(crashes[0], list):
            # Flatten nested list
            flat_crashes = crashes[0]
            return flat_crashes
        elif isinstance(crashes, list) and all(isinstance(item, dict) for item in crashes):
            return crashes

    # CASE 2: Response is a nested list [[dict, dict, ...]]
    if isinstance(data, list) and len(data) == 1 and isinstance(data[0], list):
        flat_data = data[0]
        return flat_data

    # CASE 3: Unexpected structure
    raise ValueError("Unexpected response format from API")

# ========== FUNCTION TO FETCH CRASH DETAILS ==========
def fetch_crash_details(case_year, state, state_case_id, retries=3, delay=0.5):
    url = f"{NHTSA_API_BASE}/crashes/GetCaseDetails"
    params = {
        "caseYear": case_year,
        "state": state,
        "stateCase": state_case_id,
        "format": FORMAT
    }

    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, headers=HEADERS)
            response.raise_for_status()
            data = response.json()

            if not data or not isinstance(data, dict):
                raise ValueError(f"No valid crash details for ID {state_case_id}")
            return data
        except HTTPError as e:
            if e.response.status_code == 429:  # Rate limit exceeded
                if attempt < retries - 1:
                    time.sleep(delay)
                    continue
            raise
        except Exception as e:
            logging.error(f"Error fetching details for {state_case_id}: {e}")
            raise
    raise ValueError(f"Failed to fetch details for ID {state_case_id} after {retries} attempts")

# ========== FUNCTION TO PROCESS A SINGLE CRASH ==========
def process_crash(crash, state):
    if not isinstance(crash, dict):
        return None

    state_case_id = crash.get("St_Case") or crash.get("st_case")
    case_year = crash.get("case_year") or FROM_YEAR  # Fallback to FROM_YEAR

    if not state_case_id:
        return None

    try:
        details = fetch_crash_details(case_year, state, state_case_id)

        # Navigate to CrashResultSet
        crash_data = {}
        if "Results" in details and isinstance(details["Results"], list) and details["Results"]:
            results = details["Results"][0]
            if isinstance(results, list) and results and isinstance(results[0], dict) and "CrashResultSet" in results[0]:
                crash_data = results[0]["CrashResultSet"]

        # Construct CRASH_DATE from YEAR, MONTH, DAY, HOUR, MINUTE
        crash_date = (crash_data.get("YEAR", "") + "-" + 
                      str(crash_data.get("MONTH", "")).zfill(2) + "-" + 
                      str(crash_data.get("DAY", "")).zfill(2) + " " + 
                      str(crash_data.get("HOUR", "")).zfill(2) + ":" + 
                      str(crash_data.get("MINUTE", "")).zfill(2))

        # Create record with specified fields mapped to JSON response
        record = {
            # GetCaseList fields
            "Crash ID": state_case_id,
            "CrashDate": crash.get("CrashDate", ""),
            "CountyName": crash.get("CountyName", ""),
            "Fatals": crash.get("Fatals", ""),
            "Peds": crash.get("Peds", ""),
            "Persons": crash.get("Persons", ""),
            "State": crash.get("State", ""),
            "StateName": crash.get("StateName", ""),
            "TotalVehicles": crash.get("TotalVehicles", ""),
            # GetCaseDetails fields
            "Crash Date": crash_date,
            "Latitude": crash_data.get("LATITUDE", ""),
            "Longitude": crash_data.get("LONGITUD", ""),
            "Weather": crash_data.get("WEATHER", ""),
            "Weather Name": crash_data.get("WEATHERNAME", ""),
            "Road Function": crash_data.get("ROAD_FNC", ""),
            "Road Function Name": crash_data.get("ROAD_FNCNAME", ""),
            "Light Condition": crash_data.get("LGT_COND", ""),
            "Light Condition Name": crash_data.get("LGT_CONDNAME", ""),
            "Manner of Collision": crash_data.get("MAN_COLL", ""),
            "Manner of Collision Name": crash_data.get("MAN_COLLNAME", ""),
            "Speed Limit": crash_data.get("SP_JUR", ""),
            "Speed Limit Name": crash_data.get("SP_JURNAME", ""),
            "Harmful Event": crash_data.get("HARM_EV", ""),
            "Harmful Event Name": crash_data.get("HARM_EVNAME", ""),
            "Drunk Drivers": crash_data.get("DRUNK_DR", ""),
            "City": crash_data.get("CITY", ""),
            "City Name": crash_data.get("CITYNAME", ""),
            "Functional System": crash_data.get("FUNC_SYS", ""),
            "Functional System Name": crash_data.get("FUNC_SYSNAME", ""),
            "Work Zone": crash_data.get("WRK_ZONE", ""),
            "Work Zone Name": crash_data.get("WRK_ZONENAME", "")
        }
        return record
    except Exception as e:
        logging.error(f"Error processing crash {state_case_id}: {e}")
        return None

# ========== MAIN PIPELINE FUNCTION ==========
def run_pipeline():
    for state_id, state_name in STATES:
        try:
            crashes = fetch_crash_case_list(state_id, FROM_YEAR, TO_YEAR)
            logging.info(f"Fetched {len(crashes)} crashes for {state_name}")
        except Exception as e:
            logging.error(f"Error fetching crash list for {state_name}: {e}")
            continue

        output = []

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_crash = {executor.submit(process_crash, crash, state_id): crash for crash in crashes}
            for future in tqdm(as_completed(future_to_crash), total=len(crashes), desc=f"Processing crashes for {state_name}"):
                record = future.result()
                if record:
                    output.append(record)

        if output:
            with open(f"crash_data_{state_name}.csv", "w", newline='') as file:
                writer = csv.DictWriter(file, fieldnames=output[0].keys())
                writer.writeheader()
                writer.writerows(output)

# ========== ENTRY POINT ==========
if __name__ == "__main__":
    run_pipeline()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/chardet/sbcsgroupprober.py:32: RuntimeWarning: coroutine 'run_pipeline' was never awaited
  from .langgreekmodel import ISO_8859_7_GREEK_MODEL, WINDOWS_1253_GREEK_MODEL
Processing crashes for Alabama:   1%|          | 26/5000 [00:08<16:13,  5.11it/s] 

# SUPER FAST TESTING

In [1]:
import requests
import csv
import time
import json
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.exceptions import HTTPError
import logging

# ========== CONFIGURATION ==========
NHTSA_API_BASE = "https://crashviewer.nhtsa.dot.gov/CrashAPI"
FROM_YEAR = 2020
TO_YEAR = 2023
FORMAT = "json"
HEADERS = {
    "X-Forwarded-For": "8.8.8.8",  # Simulated US IP
    "User-Agent": "Mozilla/5.0"
}
MIN_VEHICLES = 1
MAX_VEHICLES = 99
MAX_WORKERS = 100  # Increased for more concurrency

# List of all U.S. states and territories with NHTSA codes
STATES = [
    (1, "Alabama"), (2, "Alaska"), (4, "Arizona"), (5, "Arkansas"), (6, "California"),
    (8, "Colorado"), (9, "Connecticut"), (10, "Delaware"), (11, "District_of_Columbia"),
    (12, "Florida"), (13, "Georgia"), (15, "Hawaii"), (16, "Idaho"), (17, "Illinois"),
    (18, "Indiana"), (19, "Iowa"), (20, "Kansas"), (21, "Kentucky"), (22, "Louisiana"),
    (23, "Maine"), (24, "Maryland"), (25, "Massachusetts"), (26, "Michigan"), (27, "Minnesota"),
    (28, "Mississippi"), (29, "Missouri"), (30, "Montana"), (31, "Nebraska"), (32, "Nevada"),
    (33, "New_Hampshire"), (34, "New_Jersey"), (35, "New_Mexico"), (36, "New_York"),
    (37, "North_Carolina"), (38, "North_Dakota"), (39, "Ohio"), (40, "Oklahoma"), (41, "Oregon"),
    (42, "Pennsylvania"), (44, "Rhode_Island"), (45, "South_Carolina"), (46, "South_Dakota"),
    (47, "Tennessee"), (48, "Texas"), (49, "Utah"), (50, "Vermont"), (51, "Virginia"),
    (53, "Washington"), (54, "West_Virginia"), (55, "Wisconsin"), (56, "Wyoming"),
    (72, "Puerto_Rico")
]

# Set up logging
logging.basicConfig(filename="crash_data.log", level=logging.INFO)

# Create a session for connection pooling
session = requests.Session()

# ========== FUNCTION TO FETCH CRASH LIST ==========
def fetch_crash_case_list(state, from_year, to_year):
    url = f"{NHTSA_API_BASE}/crashes/GetCaseList"
    params = {
        "states": state,
        "fromYear": from_year,
        "toYear": to_year,
        "minNumOfVehicles": MIN_VEHICLES,
        "maxNumOfVehicles": MAX_VEHICLES,
        "format": FORMAT
    }

    response = session.get(url, params=params, headers=HEADERS)
    response.raise_for_status()
    data = response.json()

    # CASE 1: Response contains a 'Results' list
    if isinstance(data, dict) and "Results" in data:
        crashes = data["Results"]
        if isinstance(crashes, list) and crashes and isinstance(crashes[0], list):
            # Flatten nested list
            flat_crashes = crashes[0]
            return flat_crashes
        elif isinstance(crashes, list) and all(isinstance(item, dict) for item in crashes):
            return crashes

    # CASE 2: Response is a nested list [[dict, dict, ...]]
    if isinstance(data, list) and len(data) == 1 and isinstance(data[0], list):
        flat_data = data[0]
        return flat_data

    # CASE 3: Unexpected structure
    raise ValueError("Unexpected response format from API")

# ========== FUNCTION TO FETCH CRASH DETAILS ==========
def fetch_crash_details(case_year, state, state_case_id, retries=3, delay=0.1):
    url = f"{NHTSA_API_BASE}/crashes/GetCaseDetails"
    params = {
        "caseYear": case_year,
        "state": state,
        "stateCase": state_case_id,
        "format": FORMAT
    }

    for attempt in range(retries):
        try:
            response = session.get(url, params=params, headers=HEADERS)
            response.raise_for_status()
            data = response.json()

            if not data or not isinstance(data, dict):
                raise ValueError(f"No valid crash details for ID {state_case_id}")
            return data
        except HTTPError as e:
            if e.response.status_code == 429:  # Rate limit exceeded
                if attempt < retries - 1:
                    time.sleep(delay)
                    continue
            raise
        except Exception as e:
            logging.error(f"Error fetching details for {state_case_id}: {e}")
            raise
    raise ValueError(f"Failed to fetch details for ID {state_case_id} after {retries} attempts")

# ========== FUNCTION TO PROCESS A SINGLE CRASH ==========
def process_crash(crash, state):
    if not isinstance(crash, dict):
        return None

    state_case_id = crash.get("St_Case") or crash.get("st_case")
    case_year = crash.get("case_year") or FROM_YEAR  # Fallback to FROM_YEAR

    if not state_case_id:
        return None

    try:
        details = fetch_crash_details(case_year, state, state_case_id)

        # Navigate to CrashResultSet
        crash_data = {}
        if "Results" in details and isinstance(details["Results"], list) and details["Results"]:
            results = details["Results"][0]
            if isinstance(results, list) and results and isinstance(results[0], dict) and "CrashResultSet" in results[0]:
                crash_data = results[0]["CrashResultSet"]

        # Construct CRASH_DATE from YEAR, MONTH, DAY, HOUR, MINUTE
        crash_date = (crash_data.get("YEAR", "") + "-" + 
                      str(crash_data.get("MONTH", "")).zfill(2) + "-" + 
                      str(crash_data.get("DAY", "")).zfill(2) + " " + 
                      str(crash_data.get("HOUR", "")).zfill(2) + ":" + 
                      str(crash_data.get("MINUTE", "")).zfill(2))

        # Create record with specified fields mapped to JSON response
        record = {
            # GetCaseList fields
            "Crash ID": state_case_id,
            "CrashDate": crash.get("CrashDate", ""),
            "CountyName": crash.get("CountyName", ""),
            "Fatals": crash.get("Fatals", ""),
            "Peds": crash.get("Peds", ""),
            "Persons": crash.get("Persons", ""),
            "State": crash.get("State", ""),
            "StateName": crash.get("StateName", ""),
            "TotalVehicles": crash.get("TotalVehicles", ""),
            # GetCaseDetails fields
            "Crash Date": crash_date,
            "Latitude": crash_data.get("LATITUDE", ""),
            "Longitude": crash_data.get("LONGITUD", ""),
            "Weather": crash_data.get("WEATHER", ""),
            "Weather Name": crash_data.get("WEATHERNAME", ""),
            "Road Function": crash_data.get("ROAD_FNC", ""),
            "Road Function Name": crash_data.get("ROAD_FNCNAME", ""),
            "Light Condition": crash_data.get("LGT_COND", ""),
            "Light Condition Name": crash_data.get("LGT_CONDNAME", ""),
            "Manner of  Collision": crash_data.get("MAN_COLL", ""),
            "Manner of Collision Name": crash_data.get("MAN_COLLNAME", ""),
            "Speed Limit": crash_data.get("SP_JUR", ""),
            "Speed Limit Name": crash_data.get("SP_JURNAME", ""),
            "Harmful Event": crash_data.get("HARM_EV", ""),
            "Harmful Event Name": crash_data.get("HARM_EVNAME", ""),
            "Drunk Drivers": crash_data.get("DRUNK_DR", ""),
            "City": crash_data.get("CITY", ""),
            "City Name": crash_data.get("CITYNAME", ""),
            "Functional System": crash_data.get("FUNC_SYS", ""),
            "Functional System Name": crash_data.get("FUNC_SYSNAME", ""),
            "Work Zone": crash_data.get("WRK_ZONE", ""),
            "Work Zone Name": crash_data.get("WRK_ZONENAME", "")
        }
        return record
    except Exception as e:
        logging.error(f"Error processing crash {state_case_id}: {e}")
        return None

# ========== MAIN PIPELINE FUNCTION ==========
def run_pipeline():
    for state_id, state_name in STATES:
        crashes = []
        for year in range(FROM_YEAR, TO_YEAR + 1):
            try:
                year_crashes = fetch_crash_case_list(state_id, year, year)
                crashes.extend(year_crashes)
                logging.info(f"Fetched {len(year_crashes)} crashes for {state_name} in {year}")
            except Exception as e:
                logging.error(f"Error fetching crash list for {state_name} in {year}: {e}")
                continue

        output = []

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_crash = {executor.submit(process_crash, crash, state_id): crash for crash in crashes}
            for future in tqdm(as_completed(future_to_crash), total=len(crashes), desc=f"Processing crashes for {state_name}"):
                record = future.result()
                if record:
                    output.append(record)

        if output:
            with open(f"crash_data_{state_name}.csv", "w", newline='') as file:
                writer = csv.DictWriter(file, fieldnames=output[0].keys())
                writer.writeheader()
                writer.writerows(output)

# ========== ENTRY POINT ==========
if __name__ == "__main__":
    try:
        run_pipeline()
    finally:
        session.close()

Processing crashes for Alabama: 0it [00:00, ?it/s]
Processing crashes for Alaska: 0it [00:00, ?it/s]
Processing crashes for Arizona: 0it [00:00, ?it/s]
Processing crashes for Arkansas: 0it [00:00, ?it/s]

# Ethicial Hits

In [5]:
import requests
import csv
import time
import json
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.exceptions import HTTPError, Timeout
import logging

# ========== CONFIGURATION ==========
NHTSA_API_BASE = "https://crashviewer.nhtsa.dot.gov/CrashAPI"
FROM_YEAR = 2020
TO_YEAR = 2023
FORMAT = "json"
HEADERS = {
    "X-Forwarded-For": "8.8.8.8",  # Simulated US IP
    "User-Agent": "Mozilla/5.0"
}
MIN_VEHICLES = 1
MAX_VEHICLES = 99
MAX_WORKERS = 100  # Increased for more concurrency

# List of all U.S. states and territories with NHTSA codes
STATES = [
    (1, "Alabama")
]

# Set up logging
logging.basicConfig(filename="crash_data.log", level=logging.INFO)

# Create a session for connection pooling
session = requests.Session()

# ========== FUNCTION TO FETCH CRASH LIST ==========
def fetch_crash_case_list(state, from_year, to_year):
    time.sleep(1)  # 1-second delay between each hit
    url = f"{NHTSA_API_BASE}/crashes/GetCaseList"
    params = {
        "states": state,
        "fromYear": from_year,
        "toYear": to_year,
        "minNumOfVehicles": MIN_VEHICLES,
        "maxNumOfVehicles": MAX_VEHICLES,
        "format": FORMAT
    }

    response = session.get(url, params=params, headers=HEADERS)
    response.raise_for_status()
    data = response.json()

    # CASE 1: Response contains a 'Results' list
    if isinstance(data, dict) and "Results" in data:
        crashes = data["Results"]
        if isinstance(crashes, list) and crashes and isinstance(crashes[0], list):
            # Flatten nested list
            flat_cras mobilità

        elif isinstance(crashes, list) and all(isinstance(item, dict) for item in crashes):
            return crashes

    # CASE 2: Response is a nested list [[dict, dict, ...]]
    if isinstance(data, list) and len(data) == 1 and isinstance(data[0], list):
        flat_data = data[0]
        return flat_data

    # CASE 3: Unexpected structure
    raise ValueError("Unexpected response format from API")

# ========== FUNCTION TO FETCH CRASH DETAILS ==========
def fetch_crash_details(case_year, state, state_case_id, retries=3, delay=300):  # 5-minute delay (300 seconds)
    time.sleep(1)  # 1-second delay between each hit
    url = f"{NHTSA_API_BASE}/crashes/GetCaseDetails"
    params = {
        "caseYear": case_year,
        "state": state,
        "stateCase": state_case_id,
        "format": FORMAT
    }

    for attempt in range(retries):
        try:
            response = session.get(url, params=params, headers=HEADERS, timeout=30)  # 30-second timeout
            response.raise_for_status()
            data = response.json()

            if not data or not isinstance(data, dict):
                raise ValueError(f"No valid crash details for ID {state_case_id}")
            return data
        except HTTPError as e:
            if e.response.status_code == 429:  # Rate limit exceeded
                if attempt < retries - 1:
                    time.sleep(delay)  # 5-minute delay on retry
                    continue
            raise
        except Timeout:
            logging.error(f"API stopped responding (timeout) for crash ID {state_case_id} in state {state}, case year {case_year}")
            raise
        except Exception as e:
            logging.error(f"Error fetching details for {state_case_id}: {e}")
            raise
    raise ValueError(f"Failed to fetch details for ID {state_case_id} after {retries} attempts")

# ========== FUNCTION TO PROCESS A SINGLE CRASH ==========
def process_crash(crash, state):
    if not isinstance(crash, dict):
        return None

    state_case_id = crash.get("St_Case") or crash.get("st_case")
    case_year = crash.get("case_year") or FROM_YEAR  # Fallback to FROM_YEAR

    if not state_case_id:
        return None

    try:
        details = fetch_crash_details(case_year, state, state_case_id)

        # Navigate to CrashResultSet
        crash_data = {}
        if "Results" in details and isinstance(details["Results"], list) and details["Results"]:
            results = details["Results"][0]
            if isinstance(results, list) and results and isinstance(results[0], dict) and "CrashResultSet" in results[0]:
                crash_data = results[0]["CrashResultSet"]

        # Construct CRASH_DATE from YEAR, MONTH, DAY, HOUR, MINUTE
        crash_date = (crash_data.get("YEAR", "") + "-" + 
                      str(crash_data.get("MONTH", "")).zfill(2) + "-" + 
                      str(crash_data.get("DAY", "")).zfill(2) + " " + 
                      str(crash_data.get("HOUR", "")).zfill(2) + ":" + 
                      str(crash_data.get("MINUTE", "")).zfill(2))

        # Create record with specified fields mapped to JSON response
        record = {
            # GetCaseList fields
            "Crash ID": state_case_id,
            "CrashDate": crash.get("CrashDate", ""),
            "CountyName": crash.get("CountyName", ""),
            "Fatals": crash.get("Fatals", ""),
            "Peds": crash.get("Peds", ""),
            "Persons": crash.get("Persons", ""),
            "State": crash.get("State", ""),
            "StateName": crash.get("StateName", ""),
            "TotalVehicles": crash.get("TotalVehicles", ""),
            # GetCaseDetails fields
            "Crash Date": crash_date,
            "Latitude": crash_data.get("LATITUDE", ""),
            "Longitude": crash_data.get("LONGITUD", ""),
            "Weather": crash_data.get("WEATHER", ""),
            "Weather Name": crash_data.get("WEATHERNAME", ""),
            "Road Function": crash_data.get("ROAD_FNC", ""),
            "Road Function Name": crash_data.get("ROAD_FNCNAME", ""),
            "Light Condition": crash_data.get("LGT_COND", ""),
            "Light Condition Name": crash_data.get("LGT_CONDNAME", ""),
            "Manner of Collision": crash_data.get("MAN_COLL", ""),
            "Manner of Collision Name": crash_data.get("MAN_COLLNAME", ""),
            "Speed Limit": crash_data.get("SP_JUR", ""),
            "Speed Limit Name": crash_data.get("SP_JURNAME", ""),
            "Harmful Event": crash_data.get("HARM_EV", ""),
            "Harmful Event Name": crash_data.get("HARM_EVNAME", ""),
            "Drunk Drivers": crash_data.get("DRUNK_DR", ""),
            "City": crash_data.get("CITY", ""),
            "City Name": crash_data.get("CITYNAME", ""),
            "Functional System": crash_data.get("FUNC_SYS", ""),
            "Functional System Name": crash_data.get("FUNC_SYSNAME", ""),
            "Work Zone": crash_data.get("WRK_ZONE", ""),
            "Work Zone Name": crash_data.get("WRK_ZONENAME", "")
        }
        return record
    except Exception as e:
        logging.error(f"Error processing crash {state_case_id}: {e}")
        return None

# ========== MAIN PIPELINE FUNCTION ==========
def run_pipeline():
    for state_id, state_name in STATES:
        for year in range(FROM_YEAR, TO_YEAR + 1):
            # Create directory for the year if it doesn't exist
            year_folder = f"crash_data_{year}"
            os.makedirs(year_folder, exist_ok=True)
            
            crashes = []
            try:
                year_crashes = fetch_crash_case_list(state_id, year, year)
                crashes.extend(year_crashes)
                logging.info(f"Fetched {len(year_crashes)} crashes for {state_name} in {year}")
            except Exception as e:
                logging.error(f"Error fetching crash list for {state_name} in {year}: {e}")
                continue

            output = []

            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                future_to_crash = {executor.submit(process_crash, crash, state_id): crash for crash in crashes}
                for future in tqdm(as_completed(future_to_crash), total=len(crashes), desc=f"Processing crashes for {state_name} {year}"):
                    record = future.result()
                    if record:
                        output.append(record)

            if output:
                # Save CSV in the year-specific folder
                csv_path = os.path.join(year_folder, f"crash_data_{state_name}.csv")
                with open(csv_path, "w", newline='') as file:
                    writer = csv.DictWriter(file, fieldnames=output[0].keys())
                    writer.writeheader()
                    writer.writerows(output)

# ========== ENTRY POINT ==========
if __name__ == "__main__":
    try:
        run_pipeline()
    finally:
        session.close()

SyntaxError: invalid syntax (944008823.py, line 57)

# TESTING

In [1]:
import requests
import csv
import time
import json
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.exceptions import HTTPError
import logging

# ========== CONFIGURATION ==========
NHTSA_API_BASE = "https://crashviewer.nhtsa.dot.gov/CrashAPI"
FROM_YEAR = 2013
FORMAT = "json"
HEADERS = {
    "X-Forwarded-For": "8.8.8.8",  # Simulated US IP
    "User-Agent": "Mozilla/5.0"
}
MIN_VEHICLES = 1
MAX_VEHICLES = 99
MAX_WORKERS = 200  # Increased for more concurrency

# List of all U.S. states and territories with NHTSA codes
STATES = [
    (1, "Alabama"), (2, "Alaska"), (4, "Arizona"), (5, "Arkansas"), (6, "California"),
    (8, "Colorado"), (9, "Connecticut"), (10, "Delaware"), (11, "District_of_Columbia"),
    (12, "Florida"), (13, "Georgia"), (15, "Hawaii"), (16, "Idaho"), (17, "Illinois"),
    (18, "Indiana"), (19, "Iowa"), (20, "Kansas"), (21, "Kentucky"), (22, "Louisiana"),
    (23, "Maine"), (24, "Maryland"), (25, "Massachusetts"), (26, "Michigan"), (27, "Minnesota"),
    (28, "Mississippi"), (29, "Missouri"), (30, "Montana"), (31, "Nebraska"), (32, "Nevada"),
    (33, "New_Hampshire"), (34, "New_Jersey"), (35, "New_Mexico"), (36, "New_York"),
    (37, "North_Carolina"), (38, "North_Dakota"), (39, "Ohio"), (40, "Oklahoma"), (41, "Oregon"),
    (42, "Pennsylvania"), (44, "Rhode_Island"), (45, "South_Carolina"), (46, "South_Dakota"),
    (47, "Tennessee"), (48, "Texas"), (49, "Utah"), (50, "Vermont"), (51, "Virginia"),
    (53, "Washington"), (54, "West_Virginia"), (55, "Wisconsin"), (56, "Wyoming"),
    (72, "Puerto_Rico")
]

# Set up logging
logging.basicConfig(filename="crash_data.log", level=logging.INFO)

# Create a session for connection pooling
session = requests.Session()

# ========== FUNCTION TO FETCH CRASH LIST ==========
def fetch_crash_case_list(state, year):
    url = f"{NHTSA_API_BASE}/crashes/GetCaseList"
    params = {
        "states": state,
        "fromYear": year,
        "toYear": year,
        "minNumOfVehicles": MIN_VEHICLES,
        "maxNumOfVehicles": MAX_VEHICLES,
        "format": FORMAT
    }

    response = session.get(url, params=params, headers=HEADERS)
    response.raise_for_status()
    data = response.json()

    # CASE 1: Response contains a 'Results' list
    if isinstance(data, dict) and "Results" in data:
        crashes = data["Results"]
        if isinstance(crashes, list) and crashes and isinstance(crashes[0], list):
            # Flatten nested list
            flat_crashes = crashes[0]
            return flat_crashes
        elif isinstance(crashes, list) and all(isinstance(item, dict) for item in crashes):
            return crashes

    # CASE 2: Response is a nested list [[dict, dict, ...]]
    if isinstance(data, list) and len(data) == 1 and isinstance(data[0], list):
        flat_data = data[0]
        return flat_data

    # CASE 3: Unexpected structure
    raise ValueError("Unexpected response format from API")

# ========== FUNCTION TO FETCH CRASH DETAILS ==========
def fetch_crash_details(case_year, state, state_case_id, retries=3, delay=0.1):
    url = f"{NHTSA_API_BASE}/crashes/GetCaseDetails"
    params = {
        "caseYear": case_year,
        "state": state,
        "stateCase": state_case_id,
        "format": FORMAT
    }

    for attempt in range(retries):
        try:
            response = session.get(url, params=params, headers=HEADERS)
            response.raise_for_status()
            data = response.json()

            if not data or not isinstance(data, dict):
                raise ValueError(f"No valid crash details for ID {state_case_id}")
            return data
        except HTTPError as e:
            if e.response.status_code == 429:  # Rate limit exceeded
                if attempt < retries - 1:
                    time.sleep(delay)
                    continue
            raise
        except Exception as e:
            logging.error(f"Error fetching details for {state_case_id}: {e}")
            raise
    raise ValueError(f"Failed to fetch details for ID {state_case_id} after {retries} attempts")

# ========== FUNCTION TO PROCESS A SINGLE CRASH ==========
def process_crash(crash, state):
    if not isinstance(crash, dict):
        return None

    state_case_id = crash.get("St_Case") or crash.get("st_case")
    case_year = crash.get("case_year") or FROM_YEAR  # Fallback to FROM_YEAR

    if not state_case_id:
        return None

    try:
        details = fetch_crash_details(case_year, state, state_case_id)

        # Navigate to CrashResultSet
        crash_data = {}
        if "Results" in details and isinstance(details["Results"], list) and details["Results"]:
            results = details["Results"][0]
            if isinstance(results, list) and results and isinstance(results[0], dict) and "CrashResultSet" in results[0]:
                crash_data = results[0]["CrashResultSet"]

        # Construct CRASH_DATE from YEAR, MONTH, DAY, HOUR, MINUTE
        crash_date = (crash_data.get("YEAR", "") + "-" + 
                      str(crash_data.get("MONTH", "")).zfill(2) + "-" + 
                      str(crash_data.get("DAY", "")).zfill(2) + " " + 
                      str(crash_data.get("HOUR", "")).zfill(2) + ":" + 
                      str(crash_data.get("MINUTE", "")).zfill(2))

        # Create record with specified fields mapped to JSON response
        record = {
            # GetCaseList fields
            "Crash ID": state_case_id,
            "CrashDate": crash.get("CrashDate", ""),
            "CountyName": crash.get("CountyName", ""),
            "Fatals": crash.get("Fatals", ""),
            "Peds": crash.get("Peds", ""),
            "Persons": crash.get("Persons", ""),
            "State": crash.get("State", ""),
            "StateName": crash.get("StateName", ""),
            "TotalVehicles": crash.get("TotalVehicles", ""),
            # GetCaseDetails fields
            "Crash Date": crash_date,
            "Latitude": crash_data.get("LATITUDE", ""),
            "Longitude": crash_data.get("LONGITUD", ""),
            "Weather": crash_data.get("WEATHER", ""),
            "Weather Name": crash_data.get("WEATHERNAME", ""),
            "Road Function": crash_data.get("ROAD_FNC", ""),
            "Road Function Name": crash_data.get("ROAD_FNCNAME", ""),
            "Light Condition": crash_data.get("LGT_COND", ""),
            "Light Condition Name": crash_data.get("LGT_CONDNAME", ""),
            "Manner of Collision": crash_data.get("MAN_COLL", ""),
            "Manner of Collision Name": crash_data.get("MAN_COLLNAME", ""),
            "Speed Limit": crash_data.get("SP_JUR", ""),
            "Speed Limit Name": crash_data.get("SP_JURNAME", ""),
            "Harmful Event": crash_data.get("HARM_EV", ""),
            "Harmful Event Name": crash_data.get("HARM_EVNAME", ""),
            "Drunk Drivers": crash_data.get("DRUNK_DR", ""),
            "City": crash_data.get("CITY", ""),
            "City Name": crash_data.get("CITYNAME", ""),
            "Functional System": crash_data.get("FUNC_SYS", ""),
            "Functional System Name": crash_data.get("FUNC_SYSNAME", ""),
            "Work Zone": crash_data.get("WRK_ZONE", ""),
            "Work Zone Name": crash_data.get("WRK_ZONENAME", "")
        }
        return record
    except Exception as e:
        logging.error(f"Error processing crash {state_case_id}: {e}")
        return None

# ========== MAIN PIPELINE FUNCTION ==========
def run_pipeline():
    for state_id, state_name in STATES:
        crashes = []
        current_year = FROM_YEAR
        while current_year <= 2023:
            try:
                year_crashes = fetch_crash_case_list(state_id, current_year)
                crashes.extend(year_crashes)
                logging.info(f"Fetched {len(year_crashes)} crashes for {state_name} in {current_year}")
                current_year += 1  # Increment year after successful fetch
            except Exception as e:
                logging.error(f"Error fetching crash list for {state_name} in {current_year}: {e}")
                current_year += 1  # Move to next year on failure
                continue

        output = []

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_crash = {executor.submit(process_crash, crash, state_id): crash for crash in crashes}
            for future in tqdm(as_completed(future_to_crash), total=len(crashes), desc=f"Processing crashes for {state_name}"):
                record = future.result()
                if record:
                    output.append(record)

        if output:
            with open(f"crash_data_{state_name}.csv", "w", newline='') as file:
                writer = csv.DictWriter(file, fieldnames=output[0].keys())
                writer.writeheader()
                writer.writerows(output)

# ========== ENTRY POINT ==========
if __name__ == "__main__":
    try:
        run_pipeline()
    finally:
        session.close()

Processing crashes for Alabama:  99%|█████████▉| 9364/9416 [16:10<01:23,  1.61s/it]  